# 01 Setup and First Model Call

**Owner.** Tim (Agent Orchestration & Safety Engineer) · **Supports.** Jack

**Goal.** The environment runs and both model providers answer. At the end we have seen why a legal agent
needs more than a language model.

**Time.** About 20 minutes.

**Prerequisites.** Python 3.11 or newer and the keys stored as Deepnote environment variables (section 3).

**Definition of Done.** Connection test passes for the FHNW proxy and the Swiss AI Platform. The tool-calling
check in section 9 is documented in the decision log (task T08).

## 1. Installation

The versions are pinned in `requirements.txt`. They are the tested set from the course notebooks, so please do
not upgrade single packages. Deepnote installs `requirements.txt` automatically when the machine starts.
This cell is the fallback.

In [1]:
%pip install -q -r requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deepnote-toolkit 2.8.0 requires ipykernel<7,>=6.30.1, but you have ipykernel 7.3.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## 2. Models and endpoints

Both providers speak the OpenAI interface. That is why one class, `ChatOpenAI`, is enough for the whole project.
Only model name, address and key change.

In [2]:
# FHNW LiteLLM proxy
LLM_BASE_URL = "https://litellm.engines.aisl.science/v1"
LLM_MODEL = "azure_ai/gpt-oss-120b"     # default model for development
COMMERCIAL_MODEL = "gpt-5.6-luna"        # commercial reference model for experiment F1

# Swiss AI Platform (CSCS), hosts Apertus
SWISSAI_BASE_URL = "https://api.swissai.svc.cscs.ch/v1"
APERTUS_MODEL = None                     # detected automatically in section 5

## 3. Keys

Only the keys are secret.

- **Deepnote:** *Integrations → Environment variables*, add `LITELLM_API_KEY` and `SWISSAI_API_KEY`, then connect
  the integration to this project. The values then show up in `os.environ`.
- **Locally:** a `.env` file next to the notebook. The template is `.env.example`.
- **Missing?** The notebook asks for it.

The key is called `LITELLM_API_KEY` and not `OPENAI_API_KEY`. It belongs to the FHNW proxy, not to OpenAI.
The two only share the interface.

In [3]:
import os
from getpass import getpass
from dotenv import load_dotenv

# load_dotenv never overwrites variables that are already set.
# Order of precedence: Deepnote integration, then .env, then the input field.
load_dotenv()

for key in ["LITELLM_API_KEY", "SWISSAI_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key} (leave empty to skip): ")

HAS_SWISSAI = bool(os.environ.get("SWISSAI_API_KEY"))
print("LiteLLM key :", "set" if os.environ.get("LITELLM_API_KEY") else "MISSING")
print("Swiss AI key:", "set" if HAS_SWISSAI else "not set, Apertus checks are skipped")

LiteLLM key : set
Swiss AI key: set


## 4. Connection test

`make_llm` is the one factory we use everywhere in the project. `temperature=0` keeps answers as reproducible as
possible, which matters for the evaluation later.

In [4]:
from langchain_openai import ChatOpenAI

def make_llm(model, base_url=LLM_BASE_URL, key_env="LITELLM_API_KEY", temperature=0):
    """Create a chat model. Only model, endpoint and key change between providers."""
    return ChatOpenAI(model=model, base_url=base_url, api_key=os.environ[key_env], temperature=temperature)

llm = make_llm(LLM_MODEL)
print(llm.invoke("Answer in one sentence: which model are you?").content)

I am ChatGPT, built on OpenAI’s GPT‑4 architecture.


## 5. Which models are available?

The OpenAI interface has its own call for this. `ChatOpenAI` cannot do it, so we use the `openai` package directly.
The list also contains embedding models such as `andreasmartin/apertus-v1.1-swiss-embed-0.4b`, which we use in
notebook 02. What a model can do is not in the list. That only shows when you try it.

In [13]:
from openai import OpenAI

def list_models(base_url, key_env):
    client = OpenAI(base_url=base_url, api_key=os.environ[key_env])
    return sorted(m.id for m in client.models.list())

proxy_models = list_models(LLM_BASE_URL, "LITELLM_API_KEY")
print("FHNW LiteLLM proxy:", *proxy_models, sep="\n  ")

if HAS_SWISSAI:
    swiss_models = list_models(SWISSAI_BASE_URL, "SWISSAI_API_KEY")
    print("\nSwiss AI Platform:", *swiss_models, sep="\n  ")
    APERTUS_MODEL = next((m for m in swiss_models if m.lower().endswith("/apertus-v1.5-70b")), None)  # decision T08: Apertus-v1.5-70B (tool calling ok)

print("\nApertus model used:", APERTUS_MODEL)

FHNW LiteLLM proxy:
  all-proxy-models
  andreasmartin/apertus-v1.1-swiss-embed-0.4b
  azure_ai/claude-haiku-5-5
  azure_ai/decision-1
  azure_ai/gpt-6-luna
  azure_ai/gpt-oss-120b
  azure_ai/mistral-large-3
  gpt-5.4-mini-new
  gpt-5.6-luna
  ibm/granite4.1:3b
  leoipulsar/harrier-0.6b:latest
  tev1:0.8b

Swiss AI Platform:
  CSCS-Inference/google/gemma-4-31B-it
  CSCS-Inference/moonshotai/Kimi-K2.7-Code
  CSCS-Inference/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16
  CSCS-Inference/swiss-ai/Apertus-70B-Instruct-2509
  CSCS-Inference/swiss-ai/Apertus-8B-Instruct-2509
  CSCS-Inference/swiss-ai/Apertus-v1.5-70B
  CSCS-Inference/swiss-ai/Apertus-v1.5-70B-thinking
  CSCS-Inference/swiss-ai/Apertus-v1.5-8B
  CSCS-Inference/swiss-ai/Apertus-v1.5-8B-thinking
  CSCS-Inference/zai-org/GLM-5.2
  CSCS-Inference/zai-org/GLM-5.3
  RCP-AIaaS/BAAI/bge-reranker-v2-m3
  RCP-AIaaS/EPFLiGHT/Apertus-70B-MeditronFO
  RCP-AIaaS/OpenMeditron/Meditron3-70B
  RCP-AIaaS/OpenMeditron/meditron-CHUV-dpo
  RCP-AIa

## 6. A legal question with a known answer

The questions are in German because our sources are in German. This one has a clear answer in the law:
in the third year of service the notice period is two months to the end of a month (Art. 335c para. 1 OR).

In [6]:
question_known = (
    "Wie lange ist die Kündigungsfrist für eine Arbeitnehmerin im dritten Dienstjahr, "
    "wenn vertraglich nichts anderes vereinbart ist? Nenne den Gesetzesartikel. Antworte in 2 Sätzen."
)
print(llm.invoke(question_known).content)

Für die Arbeitnehmerin gilt, dass sie das Arbeitsverhältnis mit einer Frist von vier Wochen zum 15. oder zum Ende eines Kalendermonats kündigen kann. Diese Regelung ist in § 622 Abs. 1 BGB festgelegt.


## 7. And now something the model cannot know reliably

The next question asks for concrete court decisions with their official reference.

In [7]:
question_bge = (
    "Nenne zwei Bundesgerichtsentscheide (BGE mit Fundstelle) zur Kündigung zur Unzeit wegen Krankheit "
    "nach Art. 336c OR und fasse jeden in einem Satz zusammen."
)
print(llm.invoke(question_bge).content)

**1. BGE 4A 115/2005, 124 OR 1 (2005)**  
Das Bundesgericht erklärte die Kündigung eines Arbeitnehmers, der sich seit mehreren Monaten krankgeschrieben hatte, für unzeitgemäß, weil die krankheitsbedingte Arbeitsunfähigkeit das Vertrauen in die weitere Leistungsfähigkeit des Arbeitnehmers nachhaltig erschüttert hatte – ein Umstand, der nach Art. 336c OR die Kündigung rechtfertigen kann, jedoch nur dann, wenn die Dauer und Schwere der Erkrankung das Vertrauen in die Fortsetzung des Arbeitsverhältnisses unzumutbar machen.

**2. BGE 4A 115/2005, 124 OR 1 (2005)**  
In einem weiteren Entscheid bestätigte das Bundesgericht, dass eine Kündigung während einer akuten Krankheitsphase nur dann zulässig ist, wenn der Arbeitgeber nachweisen kann, dass die Erkrankung voraussichtlich zu einer dauerhaften oder langfristigen Arbeitsunfähigkeit führt und damit die Fortsetzung des Arbeitsverhältnisses unzumutbar macht; fehlt ein solcher Nachweis, ist die Kündigung nach Art. 336c OR unzeitgemäß.


### What just happened?

The answer sounds like a lawyer wrote it. Now check every BGE reference on bger.ch. Expect references that do
not exist or that say something else. The model did not look anything up. It produced the most plausible
continuation of the text, because a language model has no database of decisions and no way to say
"I am not sure about this reference".

This is the problem our whole project addresses:

| Notebook | What we add | What it fixes |
|---|---|---|
| 02 | Knowledge base with retrieval | Answers come from the real articles |
| 03 | MCP tools | The agent looks articles up instead of inventing them |
| 04 | Graph with router and reviewer | The process becomes visible, controllable and self-checking |
| 05 | Citation check | Invented references are blocked before the user sees them |
| 06 | Evaluation | We can measure how often it still goes wrong |

We ask `question_bge` again in notebook 03, then with a tool.

## 8. Switching the model

Only the model name changes, the rest of the code stays the same. We also measure latency and tokens. This is the
seed of experiment F1.

In [15]:
import time
import pandas as pd

def ask(model, question, base_url=LLM_BASE_URL, key_env="LITELLM_API_KEY"):
    """Ask one model one question and record latency and token usage."""
    start = time.perf_counter()
    answer = make_llm(model, base_url, key_env).invoke(question)
    usage = answer.usage_metadata or {}
    return {
        "model": model,
        "seconds": round(time.perf_counter() - start, 1),
        "input_tokens": usage.get("input_tokens"),
        "output_tokens": usage.get("output_tokens"),
        "answer": answer.content,
    }

runs = [ask(LLM_MODEL, question_known), ask(COMMERCIAL_MODEL, question_known)]
if APERTUS_MODEL:
    runs.append(ask(APERTUS_MODEL, question_known, SWISSAI_BASE_URL, "SWISSAI_API_KEY"))

results = pd.DataFrame(runs)
results[["model", "seconds", "input_tokens", "output_tokens"]]

,model,seconds,input_tokens,output_tokens
0,azure_ai/gpt-oss-120b,1.4,109,311
1,gpt-5.6-luna,1.7,48,98
2,CSCS-Inference/swiss-ai/Apertus-v1.5-70B,2.3,107,53


In [17]:
for r in runs:
    print(f"--- {r['model']} ({r['seconds']} s)\n{r['answer']}\n")

--- azure_ai/gpt-oss-120b (1.4 s)
Die gesetzliche Kündigungsfrist einer Arbeitnehmerin im dritten Dienstjahr beträgt vier Wochen zum 15. oder zum Ende eines Kalendermonats (§ 622 Abs. 1 BGB), sofern im Arbeitsvertrag nichts Abweichendes vereinbart wurde. Diese Frist gilt unabhängig von der Dauer der Betriebszugehörigkeit.

--- gpt-5.6-luna (1.7 s)
Im dritten Dienstjahr beträgt die gesetzliche Kündigungsfrist zwei Monate auf das Ende eines Monats. Rechtsgrundlage ist Art. 335c Abs. 1 OR (Schweizerisches Obligationenrecht).

--- CSCS-Inference/swiss-ai/Apertus-v1.5-70B (2.3 s)
Im dritten Dienstjahr beträgt die gesetzliche Kündigungsfrist für eine Arbeitnehmerin vier Wochen zum Monatsende, sofern vertraglich nichts anderes vereinbart ist. Dies ergibt sich aus § 622 Absatz 1 BGB.



## 9. Can the model call tools? (task T08)

Our agents depend on tool calling. For Apertus this is our biggest technical risk, so we test it first.
If a model returns no `tool_calls`, the fallback is to ask for a JSON answer and parse it ourselves.

In [19]:
from langchain_core.tools import tool

@tool
def get_article(law: str, article: str) -> str:
    """Return the text of one article of Swiss federal law, for example law='OR', article='335c'."""
    return f"(stub) text of Art. {article} {law}"

def check_tool_calling(model, base_url=LLM_BASE_URL, key_env="LITELLM_API_KEY"):
    try:
        msg = make_llm(model, base_url, key_env).bind_tools([get_article]).invoke(
            "Was steht in Art. 335c OR? Benutze das Tool."
        )
        return {"model": model, "tool_calling": bool(msg.tool_calls), "calls": msg.tool_calls}
    except Exception as error:  # some endpoints reject the tools parameter entirely
        return {"model": model, "tool_calling": False, "calls": f"error: {str(error)[:150]}"}

checks = [check_tool_calling(LLM_MODEL), check_tool_calling(COMMERCIAL_MODEL)]
if APERTUS_MODEL:
    checks.append(check_tool_calling(APERTUS_MODEL, SWISSAI_BASE_URL, "SWISSAI_API_KEY"))
pd.DataFrame(checks)

,model,tool_calling,calls
0,azure_ai/gpt-oss-120b,True,"[{'name': 'get_article', 'args': {'law': 'OR',..."
1,gpt-5.6-luna,True,"[{'name': 'get_article', 'args': {'law': 'OR',..."
2,CSCS-Inference/swiss-ai/Apertus-v1.5-70B,True,"[{'name': 'get_article', 'args': {'law': 'OR',..."


## Your turn

- Ask `question_bge` to two different models. Which one invents more? How do you notice?
- Write a question from employment or tenancy law whose answer you can check in the OR. Add it to the test set
  in notebook 06.
- Set `temperature=1` and ask the same question three times. What changes? Why do we use `0` for the evaluation?

## Decision log

| Date | Decision | Rejected alternative | Reason |
|---|---|---|---|
| 13.10.2026 | Default development model: `azure_ai/gpt-oss-120b` | `gpt-5.6-luna` as default | Both pass tool calling; gpt-oss-120b is the course default, gpt-5.6-luna stays the commercial reference for F1 |
| 13.10.2026 | Tool calling via FHNW proxy works: gpt-oss-120b and gpt-5.6-luna return correct tool_calls. Apertus: not yet tested | – | Result of section 9; Apertus needs SWISSAI_API_KEY (open, T08) |
| 13.10.2026 | Fallback if no tool calling: ask for JSON and parse it ourselves | Drop Apertus | Keeps F1 possible; same approach as the specialist schema in T10 |
| 13.10.2026 | Apertus model for agents and F1: `CSCS-Inference/swiss-ai/Apertus-v1.5-70B`. Tool calling works for gpt-oss-120b, gpt-5.6-luna, Apertus-v1.5-70B and v1.5-8B | `Apertus-70B-Instruct-2509` (picked automatically in section 5) | 2509 rejects the tools parameter (error 400, chat template); v1.5 returns correct tool_calls for get_article(OR, 335c). Set in lexgraph/config.py (PR #17, T08) |
| 13.10.2026 | Prompts anchored to Swiss law, citation check blocks §, BGB, ABGB, HGB | Rely on RAG alone | Without sources gpt-oss-120b (section 6) and Apertus v1.5 answered the notice-period question with § 622/623 BGB (PR #16) |

## Building block to take away

In [11]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

LLM_BASE_URL = "https://litellm.engines.aisl.science/v1"
SWISSAI_BASE_URL = "https://api.swissai.svc.cscs.ch/v1"

def make_llm(model="azure_ai/gpt-oss-120b", base_url=LLM_BASE_URL, key_env="LITELLM_API_KEY", temperature=0):
    return ChatOpenAI(model=model, base_url=base_url, api_key=os.environ[key_env], temperature=temperature)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=9b9d17b7-b4fd-473b-beb2-f7ed180fbdd3' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>